# 03 — Evaluate Detector

Runs inference on the test split and reports:
- mAP@0.5 and mAP@0.5:0.95
- Per-class precision / recall / F1
- Confusion matrix
- Qualitative visualisation of predictions

In [ ]:
from pathlib import Path
from ultralytics import YOLO

WEIGHTS = '../outputs/infragraph_run/weights/best.pt'
DATA    = '../configs/dataset_config.yaml'

model   = YOLO(WEIGHTS)
metrics = model.val(data=DATA, split='test', imgsz=1400)

print(f'mAP@0.5      : {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95 : {metrics.box.map:.4f}')

In [ ]:
import pandas as pd

class_names = ['router','switch','firewall','server','database','load_balancer','cloud_or_wan']

rows = []
for i, cls in enumerate(class_names):
    rows.append({
        'class':     cls,
        'precision': round(float(metrics.box.p[i]), 4),
        'recall':    round(float(metrics.box.r[i]), 4),
        'mAP50':     round(float(metrics.box.ap50[i]), 4),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt

test_imgs = list(Path('../data_generator/infragraph_dataset/images/test').glob('*.png'))
sample    = random.sample(test_imgs, min(4, len(test_imgs)))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, img_path in zip(axes.flat, sample):
    res   = model.predict(str(img_path), imgsz=1400, conf=0.25, verbose=False)[0]
    ax.imshow(res.plot()[..., ::-1])
    ax.set_title(img_path.stem, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()